# 05 — Retrieval Strategies & Failure Modes (Top-k, Scores, MMR & Filters)

**By the end of this notebook, you can inspect similarity scores, diversify results via MMR, apply metadata filters, and diagnose three classic retrieval failures.**

### Learning Objectives
- Compare `similarity_search`, `similarity_search_with_score`, and `as_retriever` with tuned `k`.
- Apply Maximal Marginal Relevance (`MMR`) to reduce duplicate overlapping chunks.
- Diagnose vague queries, vocabulary mismatch, and split answers — proving retrieval quality limits answer quality.

### Prerequisites
- Notebooks 00–04

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 00:37:38.946000 12320 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Retrieval Quality Caps Your Maximum Answer Quality

Think of your RAG pipeline like cooking dinner: if your shopper (**Retriever**) grabs rotten or missing groceries, even a 5-star chef (**LLM**) cannot cook the right meal!

- **`similarity_search_with_score`**: Shows numeric confidence so off-topic garbage gets spotted early.
- **MMR (`max_marginal_relevance_search`)**: Balances relevance to the question against *diversity* from already-selected hits.
- **Metadata Filters**: Locks search to exact subsets (`doc_type == 'changelog'`) before scoring vectors.

```
Good Q -> Crisp Chunks -> Accurate Cited Answer
Bad/Vague Q or Wrong k -> Missing Fact Chunk -> LLM Says "I Don't Know" or Guesses!
```

## 3. Step-by-Step Build

### Step 1: Index Corpus & Inspect Similarity Scores
Let's create our vector store and compare confidence scores between a direct pricing query and an irrelevant question.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=380, chunk_overlap=45).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
for q in ["Enterprise Plan monthly cost", "How to bake sourdough bread"]:
    doc, score = store.similarity_search_with_score(q, k=1)[0]
    print(f"score={score:.3f} | src={doc.metadata['source']:<20} | q={q}")
        

score=0.581 | src=company_handbook.txt | q=Enterprise Plan monthly cost
score=0.078 | src=company_handbook.txt | q=How to bake sourdough bread


> **What just happened?**
> Confidence drops cleanly from high semantic alignment down to near zero on off-topic bread recipes!

### Step 2: Wrap Store as a Retriever (`as_retriever`) & Compare MMR Diversity
Watch how `search_type='mmr'` gathers broader coverage across multiple documents on compound questions.

In [3]:
q_multi = "Enterprise Plan pricing, ZeroGravity timeout, and October 2026 HyperSync latency"
plain_srcs = [d.metadata["source"] for d in store.as_retriever(search_kwargs={"k": 4}).invoke(q_multi)]
mmr_ret = store.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.4})
mmr_srcs = [d.metadata["source"] for d in mmr_ret.invoke(q_multi)]
print("Plain Top-4 sources:", plain_srcs)
print("MMR Top-4 sources  :", mmr_srcs)
        

Plain Top-4 sources: ['company_handbook.txt', 'product_faq.md', 'recent_changelog.md', 'product_faq.md']
MMR Top-4 sources  : ['company_handbook.txt', 'product_faq.md', 'recent_changelog.md', 'recent_changelog.md']


> **What just happened?**
> MMR reduces duplicate hits from the same document and broadens multi-document coverage.

### Step 3: Apply Metadata Filters Cleanly
Using `filter=lambda doc: ...`, we force search strictly onto `recent_changelog.md` or `public` docs.

In [4]:
res = store.similarity_search("What happened in October 2026?", k=2, filter=lambda d: d.metadata.get("doc_type") == "changelog")
for d in res:
    print(f"[{d.metadata['source']}] {d.page_content[:110].replace(chr(10), ' ')}")
        

[recent_changelog.md] # Novasphere Dynamics — Product & Organization Changelog  ## Last Month Release (October 2026 — v4.6)
[recent_changelog.md] ### Edge Memory Optimization: QuantumCache v4 - Reduced resident RAM usage on NVIDIA Jetson Orin edge controll


> **What just happened?**
> Filtering excludes older handbooks even if they contain overlapping words like `'protocol'` or `'2026'`.

### Step 4: Three Retrieval Failure Modes (Why Later Notebooks Exist!)
Let's observe (1) a vague query, (2) a keyword/jargon mismatch, and (3) a tiny split-chunk miss on `eval_questions.json`.

In [5]:
tiny_store = InMemoryVectorStore.from_documents(RecursiveCharacterTextSplitter(chunk_size=110, chunk_overlap=0).split_documents(load_corpus_documents(data_dir)), embeddings)
print("1) Vague query ('policies') -> top source:", store.similarity_search("policies", k=1)[0].metadata["source"])
print("2) Jargon miss ('HTTP 410 endpoint?')  :", "410" in store.similarity_search("endpoint gone", k=1)[0].page_content)
print("3) Split answer (< 5 ms + 9872 in k=1?):", all(t in tiny_store.similarity_search("ERR_QUANTUM_TIMEOUT", k=1)[0].page_content for t in ("9872", "< 5 ms")))
        

1) Vague query ('policies') -> top source: recent_changelog.md
2) Jargon miss ('HTTP 410 endpoint?')  : True
3) Split answer (< 5 ms + 9872 in k=1?): False


> **What just happened?**
> Each failure teaches why we need Query Translation (NB08), Routing (NB09), Hybrid Indexing (NB10), and Reranking (NB11)!

## 4. Try It Yourself
1. **Check Baseline Retrieval Hit Rate@3** — *Hint:* Check how many of the 11 answerable eval questions hit the expected source inside `k=3`.
2. **Tune MMR `lambda_mult`** — *Hint:* Test `lambda_mult=0.1` vs `0.9` on `q_multi`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
import json
eval_qs = [q for q in json.loads((data_dir / "eval_questions.json").read_text(encoding="utf-8")) if q["type"] != "unanswerable"]
hits = sum(1 for q in eval_qs if any(s in d.metadata["source"] for d in store.similarity_search(q["question"], k=3) for s in (q.get("source_docs") or [q["source_doc"]])))
print(f"Baseline Hit Rate@3 = {hits}/{len(eval_qs)} ({100*hits/len(eval_qs):.1f}%)")
        

Baseline Hit Rate@3 = 11/11 (100.0%)


## 5. Common Mistakes
- Evaluating LLM prompts when the retriever never even pulled the right paragraph.
- Using `k=1` on multi-document questions — guarantees partial answers.
- Setting MMR `fetch_k` equal to `k` — leaves zero candidates for diversity selection.

## 6. Recap
- Retrieval quality strictly caps answer accuracy — always inspect retrieved chunks first.
- `similarity_search_with_score` exposes low-confidence misses.
- MMR diversity and metadata filters fix repetition and document-silo noise.

## 7. Next Notebook
Next (`06_generation_first_rag_chain.ipynb`): Wire retrieval into grounded prompts using LCEL pipes AND rebuild it cleanly as a 2-node LangGraph!

## 8. Video Chapter Outline
1. **The Retrieval Ceiling Principle** — Garbage in, garbage out.
2. **Scores & Confidence Inspecting** — Separating real matches from nonsense queries.
3. **MMR Diversity & Metadata Filters** — Broadening coverage safely.
4. **Three Retrieval Failures Live** — Setting the stage for Advanced RAG.
5. **Baseline Hit Rate@3 Check** — Measuring our starting benchmark.